## 데이터 변환 연습문제
- 타입 변환, 중복/결측 처리, 파생변수, 요약변수, 정규화/표준화

**※ 지정한 데이터 파일를 기반으로 주어진 문제 해결을 위한 코드를 작성하시오.**

필요한 모듈 임포트

### 문제#1. 이커머스 주문 데이터 (order.csv)

#### 1. 파일 로딩하여 데이터프레임 `dfA`로 생성하기

#### 2. `dfA`에서 **완전 중복 행**을 제거하고, 제거 전/후 행 수 출력하기. 중복제거한 결과는 `dfA_clean`으로 저장하기

#### 3. `dfA`의 `net_amount` 통화 문자열을 숫자 컬럼으로 변환하기
- `dfA_clean['net_amount']`에는 `₩1,234.50` 같은 문자열이 섞여 있다.
- 이를 숫자로 변환한 `net_amount_num` 컬럼 생성하기. (변환 실패는 NaN 허용)
    - 힌트: `str.replace`, `pd.to_numeric(errors='coerce')`

#### 4. dfA의 결측치 처리하기(수치/범주 혼합)
- `unit_price` 결측: **category별 중앙값**으로 대체
- `discount_rate` 결측: 0으로 대체
- `payment_type` 결측: 최빈값으로 대체
- 결과를 `dfA_clean`에 반영하기

#### 5. 4번에서 생성한 dfA_clean에서 날짜/매출 관련 파생변수 생성하기
- `order_dt`에서 `year`, `month`, `dow` 컬럼 생성 (dow : day_of_week)
- `dow`컬럼을 이용하여 토/일=1, 그 외=0를 갖는 `is_weekend` 컬럼 생성
- 단가 `unit_price`와 수량 `quantity`을 이용하여 총액 `gross_amount_calc` 컬럼 생성
- `gross_amount_calc`와 `discount_rate`컬럼을 이용하여 계산한 `net_amount_calc`컬럼 생성 : 할인률 공제한 금액

#### 6. 고객 요약변수 생성 후 원본에 결합하기
- 고객별 `order_cnt`, `net_sum`, `net_mean`, `recency_days` 생성
- `recency_days`는 **전체 최대 주문일 - 고객별 마지막 주문일** (일 단위)
- 결과 요약을 dfA_clean에 merge 하기.

#### 7. 수치 컬럼 정규화/표준화하기

- 대상: `unit_price, quantity, net_amount_calc, order_cnt, net_sum, recency_days`
- Min-Max 결과는 `*_minmax`, Z-score 결과는 `*_z`로 컬럼 생성

### 문제 #2. 학습/평가 데이터 (students.csv)

#### 1. students.csv 파일 로딩하여  데이터프레임 dfC 생성

#### 2. 중복 제거 및 attendance_rate 변환

- dfC 중복 제거 후 `dfC_clean` 생성
- `attendance_rate`의 `"92.0%"`를 0~1 실수로 변환
- 변환 후 1보다 크면 100으로 나누기

#### 3. 결측치 처리 : 그룹 대체

- `mid_score`, `final_score`: grade별 중앙값
- `study_hours`: region별 중앙값
- `attendance_rate`: 전체 중앙값
- 모든 대상 컬럼을 numeric으로 강제 변환하기

#### 4. 파생변수 생성하기 : avg_score, score_gap, is_pass, join_year/month

- `avg_score` = (mid+final)/2
- `score_gap` = final - mid
- `is_pass` = (avg_score >= 60) & (attendance_rate >= 0.85) -> 0/1
- join_dt에서 `join_year`, `join_month` 생성

#### 5. 지역-학년 요약변수 결합 + 정규화/표준화

- (region, grade)별로
  - `region_grade_avg`(avg_score 평균)
  - `region_grade_std`(avg_score 표준편차)
  - `pass_rate`(is_pass 평균)
  - `cnt`(학생 수)
- dfC_clean에 merge
- 수치 대상: `study_hours, attendance_rate, mid_score, final_score, avg_score, score_gap, region_grade_avg, pass_rate`
  - Min-Max: `*_minmax`, Z-score: `*_z`

### 문제 #3. IoT 센서 시계열 데이터 (sensor.csv)

#### 1. sensor.csv 파일 로딩하야 데이터프레임 dfB 생성하기

#### 2. 중복 제거 및 temp_c 타입 변환하기

- dfB의 중복 제거 후 `dfB_clean` 컬럼 생성
- `temp_c` 컬럼에 `'23.5C'`와 같은 데이터들을 숫자로 변환(변환 실패는 NaN)
- `ts` 기준으로 sensor_id별 정렬까지 수행

#### 3. 결측치 보간 + 잔여 결측 처리

- `hum_pct`, `vibration`을 sensor_id 그룹별로 **선형 보간**(양방향)
- 이후에도 남은 결측은 sensor_id별 중앙값으로 대체

#### 4. 6시간 롤링 요약변수 생성

- sensor_id별 `temp_roll6_mean`(6시간 이동평균)
- sensor_id별 `vib_roll6_std`(6시간 이동표준편차, NaN은 0으로)

#### 5. 일 단위 요약변수 생성 후 결합하기

- `day = ts.date` 생성
- (sensor_id, day)별로
  - `temp_day_mean`(평균)
  - `temp_day_max`(최대)
  - `hum_day_mean`(평균)
  - `vib_day_p95`(95퍼센타일)
- 결과를 dfB_clean에 merge

#### 6. 정규화/표준화 컬럼 생성하기

- 대상: `temp_c, hum_pct, vibration, temp_roll6_mean, vib_roll6_std, temp_day_mean, vib_day_p95`
- Min-Max: `*_minmax`, Z-score: `*_z`

---